# SedonaDB 0.5.0 Release


The Apache Sedona community is excited to announce the release of [SedonaDB](https://sedona.apache.org/sedonadb) version 0.5.0!

SedonaDB is the first open-source, single-node analytical database engine that treats spatial data as a first-class citizen. It is developed as a subproject of Apache Sedona. This post also covers the SedonaDB 0.4.1 changes that did not receive a separate release post.

Apache Sedona powers large-scale geospatial processing on distributed engines like Spark (SedonaSpark), Flink (SedonaFlink), and Snowflake (SedonaSnow). SedonaDB extends the Sedona ecosystem with a single-node engine optimized for small-to-medium data analytics, delivering simplicity and speed for spatial data that distributed systems and general-purpose databases cannot.

<!-- more -->

## Release Highlights

The SedonaDB 0.5.0 release features a number of exciting updates!

- FileFormat reorganization
- Python support for `DataFrame.unnest()`
- GDAL-backed raster functionality
- GeoPandas-Compatible API coverage
- Improved spatial function coverage
- Catalog extensibility

In [1]:
import sedona.db

sd = sedona.db.connect()
sd.options.interactive = True
f = sd.funcs

cities_url = "https://raw.githubusercontent.com/geoarrow/geoarrow-data/v0.2.0/natural-earth/files/natural-earth_cities.parquet"
countries_url = "https://raw.githubusercontent.com/geoarrow/geoarrow-data/v0.2.0/natural-earth/files/natural-earth_countries.parquet"

┌──────────────────────┬────────────────────┬─────────────┬────────────────────┐
│        country       ┆    dry_day_sum     ┆ pixel_years ┆   mean_dry_days    │
│         utf8         ┆       float64      ┆    int64    ┆       float64      │
╞══════════════════════╪════════════════════╪═════════════╪════════════════════╡
│ Egypt                ┆  430253.7284851074 ┆        1188 ┆  362.1664381187773 │
├╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌┼╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌┼╌╌╌╌╌╌╌╌╌╌╌╌╌┼╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌┤
│ Libya                ┆  743429.1168518066 ┆        2064 ┆  360.1885256064955 │
├╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌┼╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌┼╌╌╌╌╌╌╌╌╌╌╌╌╌┼╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌┤
│ Oman                 ┆ 124953.25396728516 ┆         348 ┆ 359.06107461863553 │
├╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌┼╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌┼╌╌╌╌╌╌╌╌╌╌╌╌╌┼╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌┤
│ United Arab Emirates ┆ 30144.409088134766 ┆          84 ┆  358.8620129539853 │
├╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌┼╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌┼╌╌╌╌╌╌╌╌╌╌╌╌╌┼╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌┤
│ Western Sahara       ┆  45

## FileFormat reorganization

In [2]:
countries = sd.read(countries_url)

sd.sql(f"""
    SELECT continent, COUNT(*) AS country_count
    FROM '{countries_url}'
    GROUP BY continent
    ORDER BY country_count DESC
""")

┌─────────────────────────┬───────────────┐
│        continent        ┆ country_count │
│           utf8          ┆     int64     │
╞═════════════════════════╪═══════════════╡
│ Africa                  ┆            51 │
├╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌┼╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌┤
│ Asia                    ┆            47 │
├╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌┼╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌┤
│ Europe                  ┆            39 │
├╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌┼╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌┤
│ North America           ┆            18 │
├╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌┼╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌┤
│ South America           ┆            13 │
├╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌┼╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌┤
│ Oceania                 ┆             7 │
├╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌┼╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌┤
│ Seven seas (open ocean) ┆             1 │
├╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌┼╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌┤
│ Antarctica              ┆             1 │
└─────────────────────────┴───────────────┘

## Python support for `DataFrame.unnest()`

In [3]:
countries = sd.read_parquet(countries_url)

country_parts = (
    countries.mutate(parts=countries.geometry.geo.dump())
    .select("name", "continent", "parts")
    .unnest("parts")
    .select("name", "continent", geometry=sd.col("parts")["geom"])
    .rename(country="name")
)

(
    country_parts.group_by("continent")
    .agg(parts=f.count(sd.col("geometry")))
    .sort(sd.col("parts").desc())
)

┌─────────────────────────┬───────┐
│        continent        ┆ parts │
│           utf8          ┆ int64 │
╞═════════════════════════╪═══════╡
│ Asia                    ┆    73 │
├╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌┼╌╌╌╌╌╌╌┤
│ Europe                  ┆    62 │
├╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌┼╌╌╌╌╌╌╌┤
│ North America           ┆    58 │
├╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌┼╌╌╌╌╌╌╌┤
│ Africa                  ┆    52 │
├╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌┼╌╌╌╌╌╌╌┤
│ Oceania                 ┆    19 │
├╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌┼╌╌╌╌╌╌╌┤
│ South America           ┆    15 │
├╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌┼╌╌╌╌╌╌╌┤
│ Antarctica              ┆     8 │
├╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌┼╌╌╌╌╌╌╌┤
│ Seven seas (open ocean) ┆     1 │
└─────────────────────────┴───────┘

## GDAL-backed raster functionality

In [6]:
import numpy as np
from sedonadb.raster import Raster

pixels = np.arange(256 * 256, dtype="float32").reshape(256, 256)
raster = Raster.from_numpy(
    pixels,
    bbox=[-180, -90, 180, 90],
    crs="OGC:CRS84",
)
np.shares_memory(pixels, raster.to_numpy())

True

Raster–vector spatial joins and zonal statistics

In [ ]:
import sedonadb_zarr

sd.register(sedonadb_zarr.ZarrExtension())
era5_url = "https://weathermapdata.rdrn.me/era5_2015_2020_l5.zarr/2"
era5_format = sedonadb_zarr.Zarr().with_options({"arrays": ["rain_ok"]})
era5 = sd.read(era5_url, format=era5_format).alias("era5")
countries = countries.alias("countries")

country_chunk_stats = (
    era5.join(
        countries,
        on=f.rs_intersects(era5.raster, countries.geometry),
    )
    .select(
        country=countries.name,
        stats=(
            era5.raster.rst.slice("year", 0).rst.zonal_stats_all(countries.geometry)
        ),
    )
)

country_totals = (
    country_chunk_stats.group_by("country")
    .agg(
        dry_day_sum=f.sum(sd.col("stats")["sum"]),
        pixel_years=f.sum(sd.col("stats")["count"]),
    )
    .filter(sd.col("pixel_years") > 0)
)

(
    country_totals.mutate(
        mean_dry_days=(
            country_totals.dry_day_sum / country_totals.pixel_years
        )
    ).sort(sd.col("mean_dry_days").desc())
)

### Raster processing

In [ ]:
sd.sql("""
WITH clipped AS (
    SELECT RS_Clip(
        RS_Example(),
        1,
        ST_GeomFromText(
            'POLYGON ((60 90, 160 90, 160 190, 60 190, 60 90))',
            'OGC:CRS84'
        )
    ) AS raster
),
tiles AS (
    SELECT UNNEST(
        RS_Tile(
            RS_Resample(raster, 32, 32, false, 'Bilinear'),
            16,
            16
        )
    ) AS raster
    FROM clipped
)
SELECT RS_Polygonize(raster, 1) AS polygons
FROM tiles
""")

### Raster construction and pixel editing

In [ ]:
sd.sql("""
WITH grid AS (
    SELECT RS_MakeEmptyRaster(
        1, 'uint8', 256, 256, ST_MakeEnvelope(0, 0, 100, 100, 4326)
    ) AS raster
),
feature AS (
    SELECT ST_SetSRID(
        ST_GeomFromText('POLYGON ((10 10, 90 10, 50 90, 10 10))'),
        4326
    ) AS geometry
)
SELECT RS_SummaryStatsAll(
    RS_SetValue(
        RS_AsRaster(geometry, raster, 'uint8', false, 1.0, 0.0, false),
        1, 1, 1, 255
    ),
    1
) AS stats
FROM grid CROSS JOIN feature
""")

## GeoPandas-Compatible API coverage

In [ ]:
import geopandas as gpd
import sedonadb_geopandas as sgpd

countries_gdf = gpd.read_parquet(countries_url)
countries_geopandas = sgpd.from_geopandas(countries_gdf).to_crs("EPSG:6933")
countries_geopandas["geometry"] = countries_geopandas.geometry.buffer(100_000)
(
    countries_geopandas.dissolve(by="continent")
    .to_crs("EPSG:4326")
    .to_geopandas()
)

## Improved spatial function coverage

Since the 0.4.0 release we have been fortunate to work with contributors to add 39 new `ST_` and `RS_` functions to our growing catalogue. These include `rs_asgeotiff`, `rs_asraster`, `rs_bandisnodata`, `rs_clip`, `rs_fromgdalraster`, `rs_geotransform`, `rs_makeemptyraster`, `rs_minconvexhull`, `rs_polygonize`, `rs_replacebandnodatavalue`, `rs_reprojectmatch`, `rs_resample`, `rs_setbandnodatavalue`, `rs_setgeoreference`, `rs_setvalue`, `rs_stack`, `rs_summarystats`, `rs_summarystatsall`, `rs_tile`, `rs_value`, `rs_values`, `rs_zonalstats`, `rs_zonalstatsall`, `st_buildarea`, `st_convexhull_agg`, `st_delaunaytriangles`, `st_exteriorring`, `st_geohash`, `st_geomcollfromtext`, `st_hausdorffdistance`, `st_linefromtext`, `st_linestringfromwkb`, `st_makeenvelope`, `st_mlinefromtext`, `st_mpointfromtext`, `st_mpolyfromtext`, `st_pointfromtext`, `st_pointfromwkb`, and `st_polygonfromtext`. This brings the SQL reference to 210 `ST_`, `RS_`, and `S2_` function pages, all of which are tested against PostGIS, SedonaSpark, and/or BigQuery for compatibility wherever possible. Check them out in our always improving [SQL reference documentation](https://sedona.apache.org/sedonadb/latest/reference/sql/)!

In [ ]:
cities = sd.read_parquet(cities_url)

(
    cities.mutate(geohash=f.st_geohash(cities.geometry, 3))
    .group_by("geohash")
    .agg(
        city_count=f.count(sd.col("name")),
        city_extent=f.st_convexhull_agg(sd.col("geometry")),
    )
    .filter(sd.col("city_count") > 1)
    .sort(sd.col("city_count").desc())
    .limit(10)
)

## Catalog extensibility

In [ ]:
import wherobots.catalog

catalog = wherobots.catalog.login()
sd.register(catalog)

sd.sql("""
    SELECT names['primary'] AS name, geometry
    FROM wherobots_open_data.overture_maps_foundation.places_place
    WHERE contains(lower(names['primary']), 'library')
      AND ST_Intersects(
          geometry,
          ST_MakeEnvelope(-71.19, 42.22, -70.92, 42.40)
      )
""").show(5)